# Lab 1 — Annotated Student Exam Performance

This notebook completes the **Student Performance in Exams** practice exercise by following the same pattern as the Palmer Penguins example.

## How to use this notebook

1. Select the **Python 3 (ipykernel)** kernel.
2. Run the cells in order the first time.
3. Read **What the result means**, then answer each **Your task** prompt in your own words.
4. Restart the kernel and use **Run All** before submission.

Short comments explain the code. Comments beginning **Added**, **Deviation**, or **Local adaptation** identify code that differs from the supplied exercise, explain why it was included, and state whether it is required or optional.

> **Learning boundary:** model interpretations are included as a reference. Your submitted explanations should be written in your own words.


## Setup — Check required packages

This setup is **added beyond the supplied code**. It installs a package only when the active kernel is missing it. This is useful for portability, but it is not part of the data analysis.


In [1]:
# Added setup: install only missing packages; optional for prepared kernels.
import importlib.util
import subprocess
import sys

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
}
missing = [
    pip_name
    for import_name, pip_name in required.items()
    if importlib.util.find_spec(import_name) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )
else:
    print("All required packages are installed.")

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)


All required packages are installed.
Python: 3.14.4
pandas: 3.0.5


**What the result means:** the active kernel has the libraries needed for inspection, clustering, and classification. Package versions can change formatting slightly, but the fixed random seeds later keep the modelling results reproducible.


## Step 1 — Select and check the dataset

Dataset: [Students Performance in Exams](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams), published on Kaggle by `spscientist`.

| Check | Result |
|---|---|
| Tabular, alpha-numeric? | Yes — a CSV containing text categories and integer scores |
| Feature count | 8 columns |
| Enough records for a first look? | Yes — 1,000 student records |

The table is based on the downloaded file rather than assumptions about the Kaggle description.

**Your task:** open the dataset page and confirm that the file structure and counts agree with this table.


## Step 2 — Document the dataset

| Field | Answer |
|---|---|
| Dataset name | Students Performance in Exams |
| Source | https://www.kaggle.com/datasets/spscientist/students-performance-in-exams |
| Access date | 23 July 2026 |
| Licence | Not stated on the supplied source |
| Original data collection | Not stated |

### Reasoned provenance note

The file may be synthetic or heavily curated teaching data, but that cannot be confirmed from the supplied documentation. It contains exactly 1,000 complete rows, generic group labels, no institution or location, no collection dates, no study method, and no source publication. Those features make it unsafe to present the rows as verified real school records.

**Your task:** explain in two or three sentences why the missing licence and collection documentation limit how confidently this dataset can be reused or generalised.


## Step 3 — Load and preview the CSV

The CSV is stored beside this notebook. `read_csv()` loads it into the required variable `df`, and `head()` previews the first five rows.


In [2]:
# Local adaptation: use the supplied file beside the notebook; required here.
import pandas as pd

df = pd.read_csv("StudentsPerformance.csv")

# Added fingerprint: detects later changes to df; optional safeguard.
source_fingerprint = pd.util.hash_pandas_object(df, index=True).sum()

# Preview the first five records.
display(df.head())


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


**What the result means:** each row represents one student record. Five columns contain background categories and three columns contain math, reading, and writing scores.

**Your task:** confirm that the preview contains the expected eight columns and readable values.


## Step 4 — Describe the structure

`shape[0]` gives rows, `shape[1]` gives columns, and `dtypes` shows how Pandas stores each column.


In [3]:
# Report dataset size.
print("Records:", df.shape[0])
print("Features:", df.shape[1])

# Show each column's storage type.
print("\nData types:")
print(df.dtypes)


Records: 1000
Features: 8

Data types:
gender                           str
race/ethnicity                   str
parental level of education      str
lunch                            str
test preparation course          str
math score                     int64
reading score                  int64
writing score                  int64
dtype: object


**What the result means:** the file has 1,000 records and 8 columns. The five background fields are text-like categories, while the three scores are integers.

**Your task:** record the row count, column count, and the difference between categorical and numeric column types.


## Step 5 — Inspect data quality

First check missing values and exact duplicate rows. A zero count does not prove the data is authentic; it only shows that these two structural problems are absent.


In [4]:
# Count gaps in each column.
print("--- Missing values ---")
print(df.isna().sum())

# Count rows that are exact copies.
print("\n--- Duplicate rows ---")
print(df.duplicated().sum())


--- Missing values ---
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

--- Duplicate rows ---
0


**What the result means:** no missing values and no exact duplicate rows are found. This is unusually tidy compared with many real operational datasets, which supports—but does not prove—the synthetic/curated interpretation.


### Inspect every categorical column

`unique()` reveals spelling variants. `value_counts()` shows whether categories are rare or imbalanced.


In [5]:
# List the five categorical columns explicitly.
categorical_cols = [
    "gender",
    "race/ethnicity",
    "parental level of education",
    "lunch",
    "test preparation course",
]

# Inspect values and counts one column at a time.
for column in categorical_cols:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].unique())
    print(df[column].value_counts())



--- gender ---
Unique values: <StringArray>
['female', 'male']
Length: 2, dtype: str
gender
female    518
male      482
Name: count, dtype: int64

--- race/ethnicity ---
Unique values: <StringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str
race/ethnicity
group C    319
group D    262
group B    190
group E    140
group A     89
Name: count, dtype: int64

--- parental level of education ---
Unique values: <StringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school']
Length: 6, dtype: str
parental level of education
some college          226
associate's degree    222
high school           196
some high school      179
bachelor's degree     118
master's degree        59
Name: count, dtype: int64

--- lunch ---
Unique values: <StringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
lunch
standard        645
free/reduced    355
Name: count, dtype: int64

--- te

**What the result means:** the categories use consistent visible spellings. Counts are not evenly distributed—for example, `none` is more common than `completed` for test preparation—so later accuracy must be compared with a majority baseline.


### Summarise scores and categories separately

Numeric scores can be meaningfully averaged. Category labels cannot, so their useful summary is count, number of unique values, most common value, and its frequency.


In [6]:
# Summarise only measured score values.
score_cols = ["math score", "reading score", "writing score"]
print("--- Numeric score summary ---")
display(df[score_cols].describe())

# Summarise category frequencies separately.
print("--- Categorical summary ---")
display(df[categorical_cols].describe())


--- Numeric score summary ---


,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


--- Categorical summary ---


,gender,race/ethnicity,parental level of education,lunch,test preparation course
count,1000,1000,1000,1000,1000
unique,2,5,6,2,2
top,female,group C,some college,standard,none
freq,518,319,226,645,642


**What the result means:** average scores are about 66.1 for math, 69.2 for reading, and 68.1 for writing. All maxima are 100; the minima are 0, 17, and 10 respectively. Keeping category summaries separate prevents meaningless calculations such as an “average” education category.

**Your task:** describe the main score ranges and explain why numeric and categorical columns require different summaries.


## Step 6 — State and test assumptions

With no data dictionary or collection method, interpretation requires assumptions. These checks test what the file can support; they cannot establish where the data came from.

1. **Score-range assumption:** all three scores use a 0–100 scale.
2. **Category-consistency assumption:** differences in case or surrounding spaces are not hiding near-duplicate labels.


In [7]:
# Check every score is between 0 and 100 inclusive.
outside_range = (
    (df[score_cols] < 0)
    | (df[score_cols] > 100)
)
print("Scores outside 0–100:")
print(outside_range.sum())

# Compare raw labels with stripped, lowercase labels.
consistency_rows = []
for column in categorical_cols:
    normalized = df[column].astype(str).str.strip().str.casefold()
    consistency_rows.append(
        {
            "column": column,
            "raw_unique": df[column].nunique(),
            "normalized_unique": normalized.nunique(),
            "consistent": df[column].nunique() == normalized.nunique(),
        }
    )

category_consistency = pd.DataFrame(consistency_rows)
print("\nCategory consistency:")
display(category_consistency)


Scores outside 0–100:
math score       0
reading score    0
writing score    0
dtype: int64

Category consistency:


,column,raw_unique,normalized_unique,consistent
0,gender,2,2,True
1,race/ethnicity,5,5,True
2,parental level of education,6,6,True
3,lunch,2,2,True
4,test preparation course,2,2,True


**What the result means:** no scores fall outside 0–100, so the stated scale is consistent with the file. Every category retains the same number of unique values after trimming spaces and ignoring case, so no simple case/whitespace near-duplicates are detected. These checks do not prove that scores were measured fairly or that the sample represents a wider student population.

**Your task:** document both assumptions, the evidence supporting them, and one limitation that the code cannot resolve.


## Step 7 — Cluster without using a label

KMeans receives only the three score columns. `StandardScaler` places them on comparable standard-deviation scales before clustering. Three clusters are used as a simple low/middle/high exploration, not because the data proves that exactly three natural groups exist.


In [8]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Work on a copy so the original df stays unchanged.
analysis_df = df.copy(deep=True)

# Scale scores so each contributes comparably.
scaler = StandardScaler()
scaled_scores = scaler.fit_transform(analysis_df[score_cols])

# Fixed settings make assignments reproducible.
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10,
)
analysis_df["cluster"] = kmeans.fit_predict(scaled_scores)

# Required comparison: clustering did not use this label.
cluster_crosstab = pd.crosstab(
    analysis_df["cluster"],
    analysis_df["test preparation course"],
)
print("Cluster by test-preparation course:")
display(cluster_crosstab)

# Added table: makes arbitrary cluster numbers interpretable; useful, not required.
print("Mean score profile for each cluster:")
display(
    analysis_df.groupby("cluster")[score_cols]
    .mean()
    .round(1)
)


Cluster by test-preparation course:


test preparation course,completed,none
cluster,,
0,154,289
1,51,198
2,153,155


Mean score profile for each cluster:


,math score,reading score,writing score
cluster,,,
0,65.3,68.5,67.8
1,48.1,50.7,48.5
2,81.7,85.1,84.2


**What the result means:** the clusters represent lower, middle, and higher score profiles, but cluster numbers themselves have no ranking until their means are inspected. The higher-score cluster has a larger proportion of `completed` students than the lower-score cluster, although both preparation categories occur in every cluster. This is an observed association, not evidence that the course caused the score differences.

**Your task:** write one sentence describing the pattern in the crosstab without making a causal claim.


## Step 8 — Predict and check accuracy

The three scores are used to predict test-preparation status. A stratified 70/30 split preserves category proportions. The decision tree follows the supplied Penguins pattern with `max_depth=4`.


In [9]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Inputs are scores; the target is preparation status.
X = analysis_df[score_cols]
y = analysis_df["test preparation course"]

# Hold back 30% for an unseen test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

# Match the shallow tree used in the supplied example.
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
model_accuracy = accuracy_score(y_test, predictions)

# Added baseline: explicitly required by Step 8.
majority_class = y_train.value_counts().idxmax()
baseline_predictions = np.full(len(y_test), majority_class)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Majority class from training data:", majority_class)
print(f"Model accuracy: {model_accuracy:.2%}")
print(f"Majority baseline: {baseline_accuracy:.2%}")
print(
    "Improvement:",
    f"{(model_accuracy - baseline_accuracy) * 100:.2f}",
    "percentage points",
)


Majority class from training data: none
Model accuracy: 65.00%
Majority baseline: 64.33%
Improvement: 0.67 percentage points


**What the result means:** the decision tree reaches approximately **65.00%** accuracy, while always guessing the training majority class reaches approximately **64.33%**. The improvement is only about **0.67 percentage points**, so the model is learning very little useful signal from the three scores. Accuracy alone would have sounded acceptable without the baseline comparison, which shows why a benchmark is necessary.

**Your task:** report both accuracies and state whether the model is genuinely learning enough to be useful.


## Final verification

These checks are **added beyond the supplied code**. They are optional safeguards that make **Run All** fail clearly if the wrong file loads or an earlier step changes unexpectedly.


In [10]:
# Added safeguards: verify the required outcomes.
final_checks = {
    "shape_is_1000_by_8": df.shape == (1000, 8),
    "no_missing_values": int(df.isna().sum().sum()) == 0,
    "no_duplicate_rows": int(df.duplicated().sum()) == 0,
    "scores_within_0_to_100": not outside_range.to_numpy().any(),
    "three_clusters": analysis_df["cluster"].nunique() == 3,
    "all_rows_clustered": len(analysis_df) == len(df),
    "source_still_unchanged": (
        source_fingerprint
        == pd.util.hash_pandas_object(df, index=True).sum()
    ),
}

print(pd.Series(final_checks, name="Passed"))
assert all(final_checks.values())


shape_is_1000_by_8        True
no_missing_values         True
no_duplicate_rows         True
scores_within_0_to_100    True
three_clusters            True
all_rows_clustered        True
source_still_unchanged    True
Name: Passed, dtype: bool


**What the result means:** every line must print `True`. These checks confirm structural requirements; they do not prove provenance, fairness, representativeness, or real-world usefulness.


## Submission checklist

- [x] Steps 1 and 2 contain completed suitability and documentation tables.
- [x] Steps 3–5 contain working code and retained output.
- [x] Step 6 documents two assumptions with supporting checks.
- [x] Step 7 contains the cluster crosstab and an interpretation.
- [x] Step 8 compares model accuracy with a majority baseline and interprets the difference.
- [ ] Rewrite the **Your task** responses in your own words.
- [ ] Restart the kernel and run all cells successfully.
- [ ] Save the final submission copy using the required filename.
